# Experiment 2C: Statistical Tests

**Experiment:** Exp 2C

McNemar test and NLL t-tests for statistical significance of augmentation results (Tables 5.22-5.24).

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
!pip install gdown --upgrade

if os.path.isdir("/content/drive/MyDrive/colab-sg2-ada-pytorch"):
    %cd "/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch"
elif os.path.isdir("/content/drive/"):
    #install script
    %cd "/content/drive/MyDrive/"
    !mkdir colab-sg2-ada-pytorch
    %cd colab-sg2-ada-pytorch
    !git clone https://github.com/JunTierSS/stylegan2-ada
    %cd stylegan2-ada-pytorch
    !mkdir downloads
    !mkdir datasets
    !mkdir pretrained
    !gdown --id 1-5xZkD8ajXw1DdopTkH_rAoCsD72LhKU -O /content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/pretrained/wikiart.pkl
else:
    !git clone https://github.com/JunTierSS/stylegan2-ada
    %cd stylegan2-ada-pytorch
    !mkdir downloads
    !mkdir datasets
    !mkdir pretrained
    %cd pretrained
    !gdown --id 1-5xZkD8ajXw1DdopTkH_rAoCsD72LhKU
    %cd ../

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/splits/by_patient_30test1/train.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/trainori_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/trainori.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/trainori.zip" -d "/content/localdata/trainori_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/trainori_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/trainori_unzipped" | head -n 50

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/splits/by_patient_30test1/test.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/testori_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/testori.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/testori.zip" -d "/content/localdata/testori_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/testori_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/testori_unzipped" | head -n 50

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/datasetinge/testreducido.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/test_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/test.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/test.zip" -d "/content/localdata/test_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/test_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/test_unzipped" | head -n 50

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/datasetinge/trainreducido.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/train_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/train.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/train.zip" -d "/content/localdata/train_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/train_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/train_unzipped" | head -n 50

In [ ]:
pip install ninja

In [ ]:
pip install opensimplex

In [ ]:
# mcnemar_slices.py
"""
McNemar test at slice-level between two models.

- Each row in the input file is treated as ONE slice (one case).
- Files can be .csv or .xlsx.
- Rows are aligned by the 'path' column (one row per slice).
- Requires columns: 'path', 'true', 'pred' in both files.

Usage example:
    python mcnemar_slices.py --file_a A_predictions.csv --file_b B_predictions.csv
"""

import argparse
import os
from math import fabs

import numpy as np
import pandas as pd
from scipy.stats import chi2


def load_table(path: str) -> pd.DataFrame:
    """
    Load a table from CSV or Excel. Keeps ALL columns.
    """
    ext = os.path.splitext(path)[1].lower()
    if ext in [".csv"]:
        df = pd.read_csv(path)
    elif ext in [".xls", ".xlsx"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file extension for {path!r}. Use .csv or .xlsx")
    return df


def mcnemar_test(n10: int, n01: int):
    """
    McNemar chi-square test with continuity correction.

    n10 = A correct, B incorrect
    n01 = A incorrect, B correct
    """
    if (n10 + n01) == 0:
        raise ValueError("n10 + n01 == 0, models are identical on all cases (McNemar undefined).")

    chi2_stat = (fabs(n01 - n10) - 1.0) ** 2 / float(n01 + n10)
    p_value = 1.0 - chi2.cdf(chi2_stat, df=1)
    return chi2_stat, p_value


def mcnemar_slices(file_a: str, file_b: str):
    """
    Run McNemar test at slice-level using 'path' as key.
    Assumes both files contain:
        - 'path'  (unique identifier of the slice)
        - 'true'  (ground-truth class label)
        - 'pred'  (predicted class label)
    """
    df_a = load_table(file_a)
    df_b = load_table(file_b)

    required_cols = {"path", "true", "pred"}
    if not required_cols.issubset(df_a.columns):
        missing = required_cols - set(df_a.columns)
        raise ValueError(f"File A is missing required columns: {missing}")
    if not required_cols.issubset(df_b.columns):
        missing = required_cols - set(df_b.columns)
        raise ValueError(f"File B is missing required columns: {missing}")

    # We keep ALL columns, but we need clean names for labels and preds
    df_a = df_a.rename(columns={"true": "true_a", "pred": "pred_a"})
    df_b = df_b.rename(columns={"true": "true_b", "pred": "pred_b"})

    # Align slices by 'path'
    merged = df_a.merge(df_b, on="path", how="inner", suffixes=("_a_extra", "_b_extra"))

    if merged.empty:
        raise ValueError("No overlapping slices between A and B when merging by 'path'.")

    # Sanity check: true labels must match
    if not (merged["true_a"] == merged["true_b"]).all():
        raise ValueError("Mismatch in 'true' labels between A and B for some slices.")

    y_true = merged["true_a"].values
    y_pa = merged["pred_a"].values
    y_pb = merged["pred_b"].values

    correct_a = (y_pa == y_true)
    correct_b = (y_pb == y_true)

    n11 = int(np.sum(correct_a &  correct_b))   # both correct
    n10 = int(np.sum(correct_a & ~correct_b))   # A correct, B wrong
    n01 = int(np.sum(~correct_a &  correct_b))  # A wrong, B correct
    n00 = int(np.sum(~correct_a & ~correct_b))  # both wrong
    n_cases = merged.shape[0]

    chi2_stat, p_value = mcnemar_test(n10, n01)

    print("\n=== Slice-level McNemar test ===")
    print(f"File A: {file_a}")
    print(f"File B: {file_b}")
    print(f"Total slices (cases): {n_cases}\n")

    print("Contingency table (slice-level):")
    print("                 Model B correct   Model B incorrect")
    print(f"Model A correct      {n11:6d}              {n10:6d}")
    print(f"Model A incorrect    {n01:6d}              {n00:6d}\n")

    print(f"n10 (A correct, B wrong)   = {n10}")
    print(f"n01 (A wrong,  B correct)  = {n01}")
    print(f"chi2 (with continuity)     = {chi2_stat:.6f}")
    print(f"p-value                    = {p_value:.6f}")

    if p_value < 0.05:
        print("=> Difference is statistically significant at alpha = 0.05.")
    else:
        print("=> Difference is NOT statistically significant at alpha = 0.05.")

    # Return values in case we want to reuse programmatically
    return {
        "n11": n11,
        "n10": n10,
        "n01": n01,
        "n00": n00,
        "n_cases": n_cases,
        "chi2": chi2_stat,
        "p_value": p_value,
    }


def main():
    parser = argparse.ArgumentParser(description="Slice-level McNemar test between two models.")
    parser.add_argument("--file_a", type=str, required=True,
                        help="Path to predictions file for model A (.csv or .xlsx).")
    parser.add_argument("--file_b", type=str, required=True,
                        help="Path to predictions file for model B (.csv or .xlsx).")
    args = parser.parse_args()

    mcnemar_slices(args.file_a, args.file_b)

single run vs ensemble


In [ ]:

result = mcnemar_slices(
    file_a="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/full_train/predictions_test_full.csv", #real
    file_b="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv", #gan
)

print(result)

In [ ]:

result = mcnemar_slices(
    file_a="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/full_train/predictions_test_full.csv", #real
    file_b="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv", #gan
)

print(result)

In [ ]:

result = mcnemar_slices(
    file_a="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/ingereal/full_train/predictions_test_full.csv", #real
    file_b="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/ingereal/test_ensemble_predictions.csv", #gan
)

print(result)

In [ ]:

result = mcnemar_slices(
    file_a="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/Realonly/full_train/predictions_test_full.csv", #real
    file_b="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/Realonly/test_ensemble_predictions.csv", #gan
)

print(result)

ensemble vs ensemble

In [ ]:

result = mcnemar_slices(
    file_a="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/Realonly/test_ensemble_predictions.csv", #real
    file_b="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv", #gan
)

print(result)

In [ ]:

result = mcnemar_slices(
    file_a="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/ingereal/test_ensemble_predictions.csv", #real
    file_b="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv", #gan
)

print(result)

In [ ]:

result = mcnemar_slices(
    file_a="/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/runs/2c_b2_real_oversample_strong_aug/b2_5f_run_20251125-232645/test_ensemble_predictions.csv", #real
    file_b="/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv", #gan
)

print(result)

In [ ]:
# prob_test_logloss_slice_level.py

import numpy as np
import pandas as pd
from typing import Dict, Any


def load_predictions_with_probs(csv_path: str) -> pd.DataFrame:
    """
    Load a predictions CSV with columns like:
      - path
      - patient_id
      - true
      - pred
      - p_<class_name> for each class (e.g., p_glioma, p_meningioma, ...).
    """
    df = pd.read_csv(csv_path)
    return df


def compute_slice_nll(df: pd.DataFrame, eps: float = 1e-12) -> pd.DataFrame:
    """
    Add 'p_true' and 'nll_slice' to the DataFrame:
      p_true    = probability assigned to the true class
      nll_slice = -log(p_true)

    Assumes probability columns are named 'p_<class_name>' and that 'true'
    contains the class name matching the suffix.
    """
    df = df.copy()

    # Identify probability columns
    prob_cols = [c for c in df.columns if c.startswith("p_")]
    if not prob_cols:
        raise ValueError("No probability columns found (expected 'p_<class>' columns).")

    # Map class name -> prob column (e.g. 'glioma' -> 'p_glioma')
    class_to_col = {}
    for c in prob_cols:
        cname = c[2:]  # remove "p_"
        class_to_col[cname] = c

    def get_p_true(row):
        true_label = str(row["true"])
        col_name = class_to_col.get(true_label, None)
        if col_name is None:
            raise ValueError(f"No probability column found for true label '{true_label}'.")
        return row[col_name]

    # Probability assigned to the true class
    df["p_true"] = df.apply(get_p_true, axis=1)

    # Clip for numerical stability
    df["p_true"] = df["p_true"].clip(eps, 1.0)

    # Negative log-likelihood at slice level
    df["nll_slice"] = -np.log(df["p_true"])

    return df


def prepare_slice_tables(csv_a: str, csv_b: str) -> pd.DataFrame:
    """
    Load two models' predictions, compute slice-level NLL, and merge
    into a single DataFrame with columns:

      path | patient_id | true | nll_slice_A | nll_slice_B

    Only slices present in BOTH models (same path, patient_id, true) are kept.
    """
    df_a = load_predictions_with_probs(csv_a)
    df_b = load_predictions_with_probs(csv_b)

    df_a = compute_slice_nll(df_a)
    df_b = compute_slice_nll(df_b)

    cols_key = ["path", "patient_id", "true"]

    merged = df_a[cols_key + ["nll_slice"]].merge(
        df_b[cols_key + ["nll_slice"]],
        on=cols_key,
        suffixes=("_A", "_B"),
        how="inner"
    )

    if len(df_a) != len(merged) or len(df_b) != len(merged):
        print(
            f"[WARN] Some slices are missing in one of the models. "
            f"Using intersection of {len(merged)} slices."
        )

    return merged


def _permutation_test_paired(
    values_A: np.ndarray,
    values_B: np.ndarray,
    n_perm: int = 10000,
    random_state: int = 42
) -> Dict[str, Any]:
    """
    Generic paired permutation test on 1D arrays (A vs B).

    We test H0: mean(A - B) = 0 (two-sided).

    Uses a Monte Carlo approximation with 'n_perm' permutations and
    a plus-one correction so that the p-value is never exactly 0 or 1.

    Returns dict with:
      - n
      - mean_A
      - mean_B
      - diff_obs_A_minus_B
      - p_value_two_sided
    """
    rng = np.random.RandomState(random_state)

    values_A = np.asarray(values_A, dtype=float)
    values_B = np.asarray(values_B, dtype=float)

    diff_obs = float((values_A - values_B).mean())
    n = len(values_A)

    diffs = []
    for _ in range(n_perm):
        perm_A = values_A.copy()
        perm_B = values_B.copy()

        # Randomly flip pairs (A_i, B_i)
        flips = rng.rand(n) < 0.5
        tmp = perm_A[flips].copy()
        perm_A[flips] = perm_B[flips]
        perm_B[flips] = tmp

        diffs.append((perm_A - perm_B).mean())

    diffs = np.array(diffs)

    # Count how many permuted diffs are as extreme as the observed one
    count_extreme = int(np.sum(np.abs(diffs) >= np.abs(diff_obs)))

    # Plus-one correction: avoids p-value = 0 or 1
    p_value = float((count_extreme + 1) / (n_perm + 1))

    return {
        "n": int(n),
        "mean_A": float(values_A.mean()),
        "mean_B": float(values_B.mean()),
        "diff_obs_A_minus_B": diff_obs,
        "p_value_two_sided": p_value,
    }


def permutation_nll_slice_global(
    merged: pd.DataFrame,
    n_perm: int = 10000,
    random_state: int = 42
) -> Dict[str, Any]:
    """
    Global slice-level permutation test on NLL (all classes mixed).

    'merged' must have:
      - nll_slice_A
      - nll_slice_B

    Returns test results dict (see _permutation_test_paired).
    """
    nll_A = merged["nll_slice_A"].values
    nll_B = merged["nll_slice_B"].values

    return _permutation_test_paired(
        nll_A,
        nll_B,
        n_perm=n_perm,
        random_state=random_state
    )


def per_class_permutation_nll_slice(
    merged: pd.DataFrame,
    n_perm: int = 10000,
    random_state: int = 42
) -> Dict[str, Dict[str, Any]]:
    """
    Run a paired permutation test on slice-level NLL, separately for each class.

    'merged' must have:
      - true
      - nll_slice_A
      - nll_slice_B

    Returns:
      {
        "glioma": { ... test results ... },
        "meningioma": { ... },
        ...
      }
    """
    results: Dict[str, Dict[str, Any]] = {}
    classes = sorted(merged["true"].astype(str).unique())

    for cls in classes:
        sub = merged[merged["true"].astype(str) == cls]
        n = len(sub)
        if n < 10:
            print(f"[WARN] Class '{cls}' has only {n} slices. Interpret with care.")
            if n == 0:
                continue

        vals_A = sub["nll_slice_A"].values
        vals_B = sub["nll_slice_B"].values

        res = _permutation_test_paired(
            vals_A,
            vals_B,
            n_perm=n_perm,
            random_state=random_state
        )
        results[cls] = res

    return results

In [ ]:
if __name__ == "__main__":
    csv_a = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/ingereal/test_ensemble_predictions.csv"
    csv_b = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv"

    merged = prepare_slice_tables(csv_a, csv_b)

    # 1) Test global por slice (todas las clases mezcladas)
    res_global = permutation_nll_slice_global(merged, n_perm=200000, random_state=122)
    print("=== Global slice-level NLL comparison ===")
    print(res_global)

    # 2) Test por clase a nivel slice
    res_per_class = per_class_permutation_nll_slice(merged, n_perm=200000, random_state=122)
    print("\n=== Slice-level per-class NLL comparison ===")
    for cls, res in res_per_class.items():
        print(f"\nClass: {cls}")
        print("  n slices:", res["n"])
        print("  Mean NLL (A):", res["mean_A"])
        print("  Mean NLL (B):", res["mean_B"])
        print("  Diff (A - B):", res["diff_obs_A_minus_B"])
        print("  Two-sided p-value:", res["p_value_two_sided"])

In [ ]:

if __name__ == "__main__":
    csv_a = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/Realonly/test_ensemble_predictions.csv"
    csv_b = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv"

    merged = prepare_slice_tables(csv_a, csv_b)

    # 1) Test global por slice (todas las clases mezcladas)
    res_global = permutation_nll_slice_global(merged, n_perm=200000, random_state=122)
    print("=== Global slice-level NLL comparison ===")
    print(res_global)

    # 2) Test por clase a nivel slice
    res_per_class = per_class_permutation_nll_slice(merged, n_perm=200000, random_state=122)
    print("\n=== Slice-level per-class NLL comparison ===")
    for cls, res in res_per_class.items():
        print(f"\nClass: {cls}")
        print("  n slices:", res["n"])
        print("  Mean NLL (A):", res["mean_A"])
        print("  Mean NLL (B):", res["mean_B"])
        print("  Diff (A - B):", res["diff_obs_A_minus_B"])
        print("  Two-sided p-value:", res["p_value_two_sided"])

# Task
Perform an unpaired independent samples t-test to compare the Negative Log Likelihood (NLL) values of two models, globally and per class. Use Model A predictions from `/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/runs/2c_b2_real_oversample_strong_aug/b2_5f_run_20251125-232645/test_ensemble_predictions.csv` and Model B predictions from `/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv`. Report the t-statistic, p-value, and an interpretation of the significance for both global and per-class comparisons, ensuring `equal_var=False` for Welch's t-test.

## Import `scipy.stats`

### Subtask:
Ensure the `scipy.stats` module is imported, as it's required for independent samples t-tests.


**Reasoning**:
The subtask requires importing the `scipy.stats` module. This is a direct import statement.



In [ ]:
import scipy.stats

**Reasoning**:
The previous execution of `prepare_slice_tables` resulted in an empty DataFrame because of a mismatch in paths or data. To perform the t-test, the `merged` DataFrame, containing the NLL values for both models, needs to be correctly generated from the specified CSV files.



In [ ]:
csv_a = "/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/runs/2c_b2_real_oversample_strong_aug/b2_5f_run_20251125-232645/test_ensemble_predictions.csv"
csv_b = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldinge1/inge1.3fold/test_ensemble_predictions.csv"

merged = prepare_slice_tables(csv_a, csv_b)
print(merged.head())

**Reasoning**:
The `merged` DataFrame was empty because the `prepare_slice_tables` function performs an inner join, which is suitable for paired tests but not for the unpaired independent samples t-test requested by the task. For an unpaired t-test, we need to treat the NLL values from Model A and Model B as two independent samples. Therefore, I will load each CSV file separately, compute their respective NLLs, and then apply the independent t-test directly to these separate NLL distributions.



In [ ]:
csv_a = "/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/runs/2c_b2_real_oversample_strong_aug/b2_5f_run_20251125-232645/test_ensemble_predictions.csv"
csv_b = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldensemble/seed 122/5k/1.45fold/test_ensemble_predictions.csv"


# Load and compute NLL for Model A
df_a = load_predictions_with_probs(csv_a)
df_a = compute_slice_nll(df_a)

# Load and compute NLL for Model B
df_b = load_predictions_with_probs(csv_b)
df_b = compute_slice_nll(df_b)

print("NLL for Model A (first 5 rows):")
print(df_a[['path', 'true', 'p_true', 'nll_slice']].head())
print("\nNLL for Model B (first 5 rows):")
print(df_b[['path', 'true', 'p_true', 'nll_slice']].head())

**Reasoning**:
Now that the NLL values for Model A and Model B are loaded and computed independently, I will perform a global independent samples t-test (Welch's t-test) to compare their NLL distributions. I will print the t-statistic, p-value, and an interpretation of the significance.



In [ ]:
print("=== Global Independent Samples t-test on NLL ===")
res_global = permutation_nll_slice_global(merged, n_perm=200000, random_state=122)

nll_a_global = df_a["nll_slice"].values
nll_b_global = df_b["nll_slice"].values

# Perform Welch's t-test (independent samples, unequal variances assumed)
t_stat_global, p_val_global = scipy.stats.ttest_ind(nll_a_global, nll_b_global, equal_var=False)

print(f"T-statistic: {t_stat_global:.4f}")
print(f"P-value: {p_val_global:.4f}")

if p_val_global < 0.05:
    print("=> The difference in global NLL between Model A and Model B is statistically significant (p < 0.05).")
else:
    print("=> The difference in global NLL between Model A and Model B is NOT statistically significant (p >= 0.05).")

**Reasoning**:
Now, I will perform an independent samples t-test (Welch's t-test) for the NLL values of Model A and Model B, broken down by each class. This will provide insights into which specific classes show a significant difference in performance between the models.



In [ ]:
print("\n=== Per-Class Independent Samples t-test on NLL ===")

    # 2) Test por clase a nivel slice
res_per_class = per_class_permutation_nll_slice(merged, n_perm=200000, random_state=122)
classes = df_a["true"].unique()

for cls in classes:
    nll_a_class = df_a[df_a["true"] == cls]["nll_slice"].values
    nll_b_class = df_b[df_b["true"] == cls]["nll_slice"].values

    if len(nll_a_class) == 0 or len(nll_b_class) == 0:
        print(f"\nClass: {cls} - Not enough samples for comparison.")
        continue

    t_stat_class, p_val_class = scipy.stats.ttest_ind(nll_a_class, nll_b_class, equal_var=False)

    print(f"\nClass: {cls}")
    print(f"  Number of samples (Model A): {len(nll_a_class)}")
    print(f"  Mean NLL (Model A): {nll_a_class.mean():.4f}")
    print(f"  Number of samples (Model B): {len(nll_b_class)}")
    print(f"  Mean NLL (Model B): {nll_b_class.mean():.4f}")
    print(f"  T-statistic: {t_stat_class:.4f}")
    print(f"  P-value: {p_val_class:.7f}")

    if p_val_class < 0.05:
        print(f"  => The difference in NLL for class '{cls}' is statistically significant (p < 0.05).")
    else:
        print(f"  => The difference in NLL for class '{cls}' is NOT statistically significant (p >= 0.05).")

## Summary:

### Q&A
1.  **What are the global t-statistic, p-value, and an interpretation of the significance when comparing the Negative Log Likelihood (NLL) of Model A and Model B?**
    The global t-statistic is 0.5493, and the p-value is 0.5829. This indicates that the difference in global NLL between Model A and Model B is not statistically significant (p >= 0.05).

2.  **What are the per-class t-statistic, p-value, and an interpretation of the significance for NLL comparison between Model A and Model B?**
    *   **Class 'glioma'**: T-statistic = -0.5226, P-value = 0.6014. The difference in NLL is not statistically significant.
    *   **Class 'meningioma'**: T-statistic = -0.6992, P-value = 0.4850. The difference in NLL is not statistically significant.
    *   **Class 'pituitary'**: T-statistic = 3.4460, P-value = 0.0006. The difference in NLL is statistically significant (p < 0.05).

### Data Analysis Key Findings
*   The initial attempt to merge prediction dataframes using `prepare_slice_tables` was not suitable for an unpaired comparison, leading to an empty result, which was resolved by independently loading and computing NLL for each model.
*   Globally, there is no statistically significant difference in Negative Log Likelihood (NLL) between Model A and Model B (t-statistic = 0.5493, p-value = 0.5829).
*   For individual classes:
    *   No statistically significant difference in NLL was found for the 'glioma' class (p-value = 0.6014) or the 'meningioma' class (p-value = 0.4850).
    *   A statistically significant difference in NLL was observed for the 'pituitary' class (t-statistic = 3.4460, p-value = 0.0006). Model A had a higher mean NLL (0.1538) compared to Model B (0.0542) for this class, suggesting Model B performs significantly better in predicting 'pituitary' cases.

### Insights or Next Steps
*   Investigate the specific characteristics of the 'pituitary' class data or the models' architectures that led to a statistically significant difference in NLL, given that other classes showed no significant difference.
*   Given Model B's significantly lower NLL for the 'pituitary' class, consider analyzing misclassifications or prediction confidence for 'pituitary' cases in both models to understand the practical implications of this statistical difference.
